Langkah 1: Persiapan & Penggabungan Data (Data Wrangling)

In [2]:
import pandas as pd
import glob

# Gabungkan semua file
files = glob.glob("*.csv")
df_list = []

for file in files:
    df = pd.read_csv(file)
    # Menambahkan kolom 'league' agar tahu data berasal dari mana
    league_name = file.split('-')[0]
    df['league'] = league_name
    df_list.append(df)

# Gabungkan menjadi satu dataframe utama
df_master = pd.concat(df_list, axis=0, ignore_index=True)

# Cek hasil penggabungan
print(f"Total baris data: {df_master.shape[0]}")
df_master.head()

Total baris data: 2756


,Unnamed: 0,id,player_name,games,time,goals,xG,assists,xA,shots,key_passes,yellow_cards,red_cards,position,team_title,npg,npxG,xGChain,xGBuildup,league
0,0,2371,Cristiano Ronaldo,33,2807,29,29.838081,3,3.854639,167,36,3,0,F S,Juventus,23,23.747810,28.635406,8.915718,Serie_A
1,1,594,Romelu Lukaku,36,2885,24,23.425768,11,8.387602,97,53,4,0,F S,Inter,18,18.857976,30.672952,5.996654,Serie_A
2,2,1229,Luis Muriel,36,1434,22,16.692218,8,4.887805,86,46,0,0,F M S,Atalanta,20,14.408361,23.752104,6.238759,Serie_A
3,3,7084,Dusan Vlahovic,37,2945,21,18.476590,3,2.296164,86,18,1,0,F S,Fiorentina,15,13.908837,14.827192,1.562626,Serie_A
4,4,1209,Ciro Immobile,35,2887,20,19.771116,6,7.202586,119,54,3,1,F S,Lazio,16,13.680725,24.116897,4.969870,Serie_A


Langkah 2: Simulasi SQL (Analisis Performa)

In [3]:
# Menggunakan logic 'RANK' (seperti SQL Window Function)
df_master['rank'] = df_master.groupby('league')['goals'].rank(method='dense', ascending=False)

# Memfilter hanya Top 3 per liga
top_3_per_league = df_master[df_master['rank'] <= 3][['league', 'player_name', 'goals', 'rank']]
top_3_per_league = top_3_per_league.sort_values(by=['league', 'rank'])

print(top_3_per_league)

          league           player_name  goals  rank
1169  Bundesliga    Robert Lewandowski     41   1.0
1170  Bundesliga           André Silva     28   2.0
1171  Bundesliga        Erling Haaland     27   3.0
1665      LaLiga          Lionel Messi     30   1.0
1666      LaLiga         Gerard Moreno     23   2.0
1667      LaLiga         Karim Benzema     23   2.0
1668      LaLiga           Luis Suárez     21   3.0
594      Ligue_1  Kylian Mbappe-Lottin     27   1.0
595      Ligue_1         Memphis Depay     21   2.0
596      Ligue_1     Wissam Ben Yedder     20   3.0
0        Serie_A     Cristiano Ronaldo     29   1.0
1        Serie_A         Romelu Lukaku     24   2.0
2        Serie_A           Luis Muriel     22   3.0
2234         epl            Harry Kane     23   1.0
2235         epl         Mohamed Salah     22   2.0
2236         epl       Bruno Fernandes     18   3.0


Langkah 3: Analisis Efisiensi (Business Insight)

In [4]:
# Menghindari pembagian dengan nol
df_master['conversion_rate'] = df_master.apply(
    lambda row: (row['goals'] / row['shots']) if row['shots'] > 0 else 0, axis=1
)

# Fokus pada pemain yang punya menit bermain > 500 (untuk hasil yang signifikan)
df_efficient = df_master[df_master['time'] > 500].sort_values(by='conversion_rate', ascending=False)

# Lihat 5 pemain paling efisien
print(df_efficient[['player_name', 'goals', 'shots', 'conversion_rate']].head(5))

           player_name  goals  shots  conversion_rate
1887              Bono      1      1         1.000000
2432           Alisson      1      1         1.000000
823    Ambroise Oyongo      1      1         1.000000
2367         Issa Diop      2      3         0.666667
1791  Álvaro Odriozola      2      3         0.666667


Langkah 4: Export untuk Dashboard

In [5]:
# Export ke CSV untuk di-upload ke Google Sheets/Looker Studio
df_master.to_csv('final_data_analisis.csv', index=False)
from google.colab import files
files.download('final_data_analisis.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>